# 제6장. 시스템 다이내믹스

## 학습 목표
1. 선형적 사고의 한계를 인식하고, 시스템 사고의 필요성을 설명할 수 있다
2. 강화 루프(R)와 균형 루프(B)를 구분하고, 피드백 구조를 분석할 수 있다
3. 스톡-플로우 모델을 이해하고 시뮬레이션 결과를 해석할 수 있다
4. 인과 루프 다이어그램(CLD)을 작성하고 레버리지 포인트를 식별할 수 있다
5. 정책 시나리오를 시뮬레이션하여 의도치 않은 결과를 예측할 수 있다

---

### 강의 구조 (3시간)

| 시간 | 구분 | 내용 |
|------|------|------|
| | Part 1 | 시스템 사고: 선형적 사고의 한계와 피드백 루프 |
| | Part 2 | 스톡-플로우와 인과 루프 다이어그램(CLD) |
| | 휴식 | |
| | Part 3 | 레버리지 포인트와 의도치 않은 결과 |
| | Part 4 | 실습: CLD 시각화와 루프 분석 |
| | Part 5 | 실습: SaaS 성장 시뮬레이션과 정책 비교 |

---

In [1]:
# ============================================================
# 환경설정 및 라이브러리 로드
# ============================================================
# 처음 사용 시 아래 명령을 터미널에서 먼저 실행하세요:
#   python setup_env.py
# 그 후 VSCode 우측 상단에서 커널을 'AI 기획 강의 (Python 3)'으로 선택하세요.
# ============================================================

import sys

# 패키지 확인
_required = ["numpy", "pandas", "plotly"]
_missing = []
for _pkg in _required:
    try:
        __import__(_pkg)
    except ImportError:
        _missing.append(_pkg)
if _missing:
    print(f"❌ 누락된 패키지: {', '.join(_missing)}")
    print("   터미널에서 다음 명령을 실행하세요: python setup_env.py")
    raise SystemExit(1)

# 라이브러리 임포트
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ 라이브러리 로드 완료!")

✅ 라이브러리 로드 완료!


---

## Part 1. 시스템 사고: 선형적 사고의 한계와 피드백 루프

---

### 6.1 왜 시스템 사고가 필요한가?

기획자들이 가장 흔하게 저지르는 오류는 **복잡한 문제를 선형적으로 사고**하는 것이다.

> "A를 하면 B가 된다" — 이 단순한 가정이 얼마나 많은 정책과 전략을 실패하게 만들었는가?

### 교통 혼잡의 역설 (유발 수요)

```
선형적 사고:   도로 혼잡 → 도로 확장 → 혼잡 해소 ✓

실제 시스템:   도로 혼잡 → 도로 확장 → 운전 편리 → 자가용 증가
                                        → 대중교통 이용 감소
                                        → 대중교통 서비스 축소
                                        → 더 많은 자가용 전환
                                        → 도로 다시 혼잡! ✗
```

| 구분 | 선형적 사고 | 시스템 사고 |
|------|------------|------------|
| 인과관계 | 일방향 (A→B) | 순환 (A→B→...→A) |
| 시간 관점 | 정적, 단기 | 동적, 장기 |
| 분석 단위 | 개별 요소 | 전체 시스템 |
| 최적화 | 부분 최적화 | 전체 최적화 |
| 대응 방식 | 증상 치료 | 근본 구조 변화 |

In [2]:
# 시각화: 선형적 사고 vs 시스템 사고
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Linear Thinking', 'Systems Thinking'),
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}]]
)

# 선형적 사고: 직선 화살표
linear_nodes_x = [0, 1, 2]
linear_nodes_y = [1, 1, 1]
linear_labels = ['Road<br>Congestion', 'Road<br>Expansion', 'Congestion<br>Resolved']
linear_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

fig.add_trace(go.Scatter(
    x=linear_nodes_x, y=linear_nodes_y,
    mode='markers+text',
    marker=dict(size=50, color=linear_colors),
    text=linear_labels,
    textposition='bottom center',
    textfont=dict(size=10),
    showlegend=False
), row=1, col=1)

for i in range(2):
    fig.add_annotation(
        x=linear_nodes_x[i+1]-0.15, y=1,
        ax=linear_nodes_x[i]+0.15, ay=1,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowsize=1.5, arrowcolor='gray'
    )

# 시스템 사고: 순환 구조
n_nodes = 5
angles = np.linspace(0, 2*np.pi, n_nodes, endpoint=False) - np.pi/2
radius = 0.8
cx, cy = 1.0, 1.0
sys_x = cx + radius * np.cos(angles)
sys_y = cy + radius * np.sin(angles)
sys_labels = ['Road<br>Congestion', 'Road<br>Expansion', 'Driving<br>Convenience',
              'More<br>Cars', 'Transit<br>Decline']
sys_colors = ['#FF6B6B', '#4ECDC4', '#FFEAA7', '#E17055', '#DDA0DD']

fig.add_trace(go.Scatter(
    x=sys_x, y=sys_y,
    mode='markers+text',
    marker=dict(size=40, color=sys_colors),
    text=sys_labels,
    textposition=['top center', 'middle right', 'bottom center',
                  'bottom center', 'middle left'],
    textfont=dict(size=9),
    showlegend=False
), row=1, col=2)

for i in range(n_nodes):
    ni = (i + 1) % n_nodes
    fig.add_annotation(
        x=sys_x[ni], y=sys_y[ni],
        ax=sys_x[i], ay=sys_y[i],
        xref='x2', yref='y2', axref='x2', ayref='y2',
        showarrow=True, arrowhead=2, arrowsize=1.2, arrowcolor='gray'
    )

fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False)
fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False)
fig.update_layout(height=400, title_text='Linear Thinking vs Systems Thinking')
fig.show()

print("💡 핵심: 선형적 사고는 A→B만 본다. 시스템 사고는 A→B→...→A 순환을 본다.")
print("   도로 확장 → 유발 수요(Induced Demand) → 다시 혼잡. 이것이 피드백 루프다.")

💡 핵심: 선형적 사고는 A→B만 본다. 시스템 사고는 A→B→...→A 순환을 본다.
   도로 확장 → 유발 수요(Induced Demand) → 다시 혼잡. 이것이 피드백 루프다.


### 6.1.2 두 가지 피드백 루프

시스템의 동태를 결정하는 두 가지 핵심 구조:

| 특성 | 강화 루프 (Reinforcing, R) | 균형 루프 (Balancing, B) |
|------|--------------------------|------------------------|
| 피드백 | 양의 피드백 | 음의 피드백 |
| 변화 방향 | 증폭 (같은 방향) | 억제 (반대 방향) |
| 결과 | 지수적 성장 또는 붕괴 | 목표 수렴, 안정화 |
| 식별법 | 음의 극성(-)이 **짝수**개 | 음의 극성(-)이 **홀수**개 |
| 비유 | 눈덩이 효과 | 온도조절기 |

**강화 루프 예시 — 구전 마케팅:**
```
고객 수 ──(+)──→ 구전 확산 ──(+)──→ 신규 고객 ──(+)──→ 고객 수 ↑
```
모든 연결이 (+)이므로 음의 극성 0개(짝수) → **강화 루프(R)**

**균형 루프 예시 — 온도조절기:**
```
목표 온도 - 현재 온도 = 격차 ──(+)──→ 가열 ──(+)──→ 현재 온도 ──(-)──→ 격차 ↓
```
음의 극성 1개(홀수) → **균형 루프(B)**

In [3]:
# 시뮬레이션: 강화 루프 vs 균형 루프의 동태
months = 24
t = np.arange(months)

# 강화 루프: 지수 성장 (구전 마케팅)
growth_rate = 0.08  # 월 8% 성장
reinforcing_growth = 100 * (1 + growth_rate) ** t

# 강화 루프: 지수 붕괴 (역방향)
decline_rate = 0.06
reinforcing_decline = 500 * (1 - decline_rate) ** t

# 균형 루프: 목표 수렴 (온도조절기)
target = 300
initial = 100
adjustment_speed = 0.15
balancing = np.zeros(months)
balancing[0] = initial
for i in range(1, months):
    gap = target - balancing[i-1]
    balancing[i] = balancing[i-1] + adjustment_speed * gap

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        'Reinforcing Loop: Growth',
        'Reinforcing Loop: Decline',
        'Balancing Loop: Goal-Seeking'
    )
)

fig.add_trace(go.Scatter(
    x=t, y=reinforcing_growth,
    mode='lines', line=dict(color='#2ecc71', width=3),
    name='Growth (R)'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=t, y=reinforcing_decline,
    mode='lines', line=dict(color='#e74c3c', width=3),
    name='Decline (R)'
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=t, y=balancing,
    mode='lines', line=dict(color='#3498db', width=3),
    name='Converge (B)'
), row=1, col=3)
fig.add_trace(go.Scatter(
    x=t, y=[target]*months,
    mode='lines', line=dict(color='orange', width=2, dash='dash'),
    name='Target'
), row=1, col=3)

fig.update_xaxes(title_text='Month')
fig.update_yaxes(title_text='Value', col=1)
fig.update_layout(height=350, title_text='Two Types of Feedback Loops')
fig.show()

print("💡 강화 루프(R): 변화를 증폭한다 → 지수 성장 또는 지수 붕괴")
print("   균형 루프(B): 변화를 억제한다 → 목표에 수렴 (안정화)")
print("\n⚠️ 강화 루프의 위험: 성장을 만들던 루프가 역방향으로 돌면 쇠퇴도 가속화된다!")

💡 강화 루프(R): 변화를 증폭한다 → 지수 성장 또는 지수 붕괴
   균형 루프(B): 변화를 억제한다 → 목표에 수렴 (안정화)

⚠️ 강화 루프의 위험: 성장을 만들던 루프가 역방향으로 돌면 쇠퇴도 가속화된다!


### 6.1.3 지연(Delay)의 영향

**지연(Delay)**은 피드백 효과가 즉시 나타나지 않고 시차를 두고 나타나는 현상이다.

| 지연 유형 | 원인 | 결과 | 예시 |
|----------|------|------|------|
| 정보 지연 | 보고, 측정 | 늦은 대응 | 고객 이탈 감지 지연 |
| 물질 지연 | 생산, 운송 | 오버슈팅 | 재고 과잉/과소 |
| 의사결정 지연 | 숙의, 승인 | 기회 상실 | 시장 대응 지연 |

**온수 샤워 비유:** 물이 차가우면 밸브를 더 연다. 배관 지연 때문에 아직 차가운 물이 나온다. 더 연다. 갑자기 뜨거운 물이 쏟아진다! → **오버슈팅(Overshooting)**

In [4]:
# 시뮬레이션: 지연이 만드는 진동 (온도 조절 비유)
steps = 60
target_temp = 40.0  # 목표 온도

def simulate_with_delay(delay_steps, adjustment_rate=0.3):
    """지연이 있는 균형 루프 시뮬레이션"""
    temp = np.zeros(steps)
    temp[0] = 20.0  # 초기 온도
    actions = np.zeros(steps)  # 실제 적용되는 조절량
    pending = []  # 지연된 조절
    
    for i in range(1, steps):
        # 지연된 조절 적용
        applied = sum(a for t, a in pending if t <= i)
        pending = [(t, a) for t, a in pending if t > i]
        
        # 현재 온도에 기반한 조절 (지연 존재)
        gap = target_temp - temp[i-1]
        action = adjustment_rate * gap
        pending.append((i + delay_steps, action))
        
        # 온도 변화 (자연 냉각 포함)
        temp[i] = temp[i-1] + applied - 0.05 * (temp[i-1] - 20)  # 약간의 냉각
    
    return temp

# 지연 없음 vs 짧은 지연 vs 긴 지연
temp_no_delay = simulate_with_delay(0)
temp_short_delay = simulate_with_delay(3)
temp_long_delay = simulate_with_delay(8)

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(steps)), y=temp_no_delay,
    mode='lines', name='No Delay', line=dict(color='#2ecc71', width=2)))
fig.add_trace(go.Scatter(x=list(range(steps)), y=temp_short_delay,
    mode='lines', name='Short Delay (3 steps)', line=dict(color='#f39c12', width=2)))
fig.add_trace(go.Scatter(x=list(range(steps)), y=temp_long_delay,
    mode='lines', name='Long Delay (8 steps)', line=dict(color='#e74c3c', width=2)))
fig.add_hline(y=target_temp, line_dash='dash', line_color='gray',
              annotation_text='Target')

fig.update_layout(
    title='Effect of Delay on System Behavior (Shower Temperature Analogy)',
    xaxis_title='Time Step',
    yaxis_title='Temperature',
    height=400
)
fig.show()

print("💡 지연이 길수록 시스템은 불안정해진다:")
print("   - 지연 없음: 부드럽게 목표에 수렴")
print("   - 짧은 지연: 약간의 오버슈팅 후 안정")
print("   - 긴 지연: 진동(Oscillation) 발생 → 과잉 반응의 반복")
print("\n📌 채찍효과(Bullwhip Effect): 소비자 수요 10% 변화 → 공급망 끝에서 25% 변동!")

💡 지연이 길수록 시스템은 불안정해진다:
   - 지연 없음: 부드럽게 목표에 수렴
   - 짧은 지연: 약간의 오버슈팅 후 안정
   - 긴 지연: 진동(Oscillation) 발생 → 과잉 반응의 반복

📌 채찍효과(Bullwhip Effect): 소비자 수요 10% 변화 → 공급망 끝에서 25% 변동!


### 6.1.4 창발적 현상

**창발(Emergence)**: 개별 구성요소의 행동으로는 예측할 수 없는 전체 시스템의 새로운 특성

- **새 떼의 군무**: 개별 새는 단순 규칙(거리 유지, 방향 정렬)만 따르지만, 전체는 놀랍도록 복잡한 패턴
- **교통 정체**: 개인들의 합리적 선택(자가용)이 집단적 비합리(정체)를 만듦
- **조직 비효율**: 모든 부서가 자기 예산 극대화 → 전체 조직 효율성 하락

> **"부분의 합 ≠ 전체"** — 이것이 시스템 사고의 핵심

### 시스템 사고의 선구자들

| 학자 | 핵심 저서 | 공헌 |
|------|----------|------|
| Jay Forrester | Industrial Dynamics (1961) | 시스템 다이내믹스 창시 |
| Donella Meadows | The Limits to Growth (1972) | 환경 문제 적용, 레버리지 포인트 |
| Peter Senge | The Fifth Discipline (1990) | 조직 학습과 연결, 경영학 대중화 |

### 📝 이론 과제 6-1 (15분)

#### 과제: 피드백 루프 식별

**질문:**

1. **강화 루프(R)와 균형 루프(B)**를 각각 일상 생활에서 하나씩 찾아 설명하시오. 루프를 구성하는 변수들과 극성(+/-)을 명시하시오. (3-4문장)

2. **항생제 남용 문제**를 피드백 루프로 설명하시오. 어떤 유형의 루프(R/B)가 관여하며, 지연은 어떤 역할을 하는가? (3-4문장)

3. Peter Senge의 『학습하는 조직(The Fifth Discipline)』에서 **시스템 사고가 왜 다섯 번째 역량**인지 검색하여 설명하시오. (2-3문장)

**조사 키워드:**
- "reinforcing loop example"
- "balancing loop example"
- "Peter Senge fifth discipline systems thinking"

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

### ✅ 과제 6-1 제출란

**학번:** ___________
**이름:** ___________

#### 1. 강화 루프(R) 사례

(여기에 작성)

#### 1. 균형 루프(B) 사례

(여기에 작성)

#### 2. 항생제 남용의 피드백 구조

(여기에 작성)

#### 3. 시스템 사고가 다섯 번째 역량인 이유

(여기에 작성)

**출처:** (URL)

---

## Part 2. 스톡-플로우와 인과 루프 다이어그램

---

### 6.2 스톡과 플로우

**스톡(Stock)**: 시스템에 축적되는 것 (물탱크의 물, 은행 잔액, 고객 수)

**플로우(Flow)**: 스톡의 변화율 (유입/유출, 수입/지출, 고객 유입/이탈)

```
       [유입 Inflow]                   [유출 Outflow]
            ↓                               ↓
       ━━━╋━━━━━━━━ [  Stock  ] ━━━━━━━━╋━━━
            밸브                           밸브
```

**핵심 수식:**

$$Stock(t) = Stock(t-1) + Inflow(t) - Outflow(t)$$

$$\frac{dStock}{dt} = Inflow - Outflow$$

**스톡의 중요한 특성 — 관성(Inertia):**
- 플로우가 갑자기 변해도 스톡은 천천히 변한다
- 이것이 시스템의 "기억"이며, 변화에 저항하는 원인

In [5]:
# 시뮬레이션: 스톡-플로우 기본 모델 (물탱크 비유)
months = 36
t = np.arange(months)

# 시나리오: 유입이 변하는 경우
inflow = np.where(t < 12, 10, np.where(t < 24, 5, 15))  # 유입 변동
outflow = np.full(months, 8.0)  # 일정한 유출

stock = np.zeros(months)
stock[0] = 100  # 초기 스톡
for i in range(1, months):
    stock[i] = stock[i-1] + inflow[i] - outflow[i]

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Stock (Accumulated Level)', 'Flows (Rate of Change)'),
    shared_xaxes=True,
    vertical_spacing=0.12
)

# 스톡
fig.add_trace(go.Scatter(
    x=t, y=stock, mode='lines', name='Stock',
    line=dict(color='#3498db', width=3),
    fill='tozeroy', fillcolor='rgba(52, 152, 219, 0.2)'
), row=1, col=1)

# 플로우
fig.add_trace(go.Scatter(
    x=t, y=inflow, mode='lines+markers', name='Inflow',
    line=dict(color='#2ecc71', width=2)
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=t, y=outflow, mode='lines+markers', name='Outflow',
    line=dict(color='#e74c3c', width=2, dash='dash')
), row=2, col=1)

fig.update_xaxes(title_text='Month', row=2, col=1)
fig.update_yaxes(title_text='Level', row=1, col=1)
fig.update_yaxes(title_text='Flow Rate', row=2, col=1)
fig.update_layout(height=500, title_text='Stock-Flow Model: Water Tank Analogy')
fig.show()

print("💡 관찰:")
print(f"   - 유입 > 유출 (Month 0-11): 스톡 증가 ({stock[0]:.0f} → {stock[11]:.0f})")
print(f"   - 유입 < 유출 (Month 12-23): 스톡 감소 ({stock[11]:.0f} → {stock[23]:.0f})")
print(f"   - 유입 > 유출 (Month 24-35): 스톡 다시 증가 ({stock[23]:.0f} → {stock[35]:.0f})")
print("\n📌 핵심: 스톡은 플로우가 변해도 즉시 변하지 않는다 (관성). 이것이 지연의 원인!")

💡 관찰:
   - 유입 > 유출 (Month 0-11): 스톡 증가 (100 → 122)
   - 유입 < 유출 (Month 12-23): 스톡 감소 (122 → 86)
   - 유입 > 유출 (Month 24-35): 스톡 다시 증가 (86 → 170)

📌 핵심: 스톡은 플로우가 변해도 즉시 변하지 않는다 (관성). 이것이 지연의 원인!


### 6.3 인과 루프 다이어그램 (CLD)

**인과 루프 다이어그램(Causal Loop Diagram, CLD)**은 시스템의 피드백 구조를 시각화하는 도구이다.

#### CLD 표기법

| 요소 | 표기 | 의미 |
|------|------|------|
| 변수 | 텍스트 | 시스템의 상태 또는 행동 |
| 양의 인과 | →(+) | 같은 방향으로 변화 |
| 음의 인과 | →(-) | 반대 방향으로 변화 |
| 강화 루프 | R | 증폭 피드백 |
| 균형 루프 | B | 억제 피드백 |
| 지연 | ∥ | 시간 지연 |

#### 루프 유형 판별법

```python
def identify_loop_type(polarities):
    """루프 유형 식별: 음의 극성 개수로 판별"""
    negative_count = polarities.count("-")
    return "R (Reinforcing)" if negative_count % 2 == 0 else "B (Balancing)"
```

#### CLD 예시: SaaS 스타트업 성장과 한계

```
[R1: 성장 엔진]              [B1: 성장 한계]
고객 수 ─(+)─→ 구전 효과      고객 수 ─(+)─→ 서버 부하
  ↑              │                              │
  │  (+)         ↓ (+)                         ↓ (-)
  └───── 신규 고객            서비스 품질 ←──────┘
                                   │
                                   ↓ (+)
                              고객 만족도
                                   │
                                   ↓ (-)
                              고객 이탈 ─(-)─→ 고객 수
```

- **R1 지배**: 초기 빠른 성장
- **B1 강화**: 고객 증가 → 부하 → 품질↓ → 이탈↑
- **전환점**: B1이 R1을 압도 → 성장 멈춤 ("성장의 한계" 아키타입)

In [6]:
# CLD 시각화: SaaS 스타트업 성장 모델
def draw_cld(nodes, edges, title, loop_labels=None):
    """인과 루프 다이어그램을 Plotly로 시각화
    
    Args:
        nodes: dict {name: (x, y)} 노드 위치
        edges: list of (from, to, polarity) 연결
        title: 그래프 제목
        loop_labels: list of (x, y, text) 루프 레이블
    """
    fig = go.Figure()
    
    # 엣지 그리기
    for src, tgt, polarity in edges:
        x0, y0 = nodes[src]
        x1, y1 = nodes[tgt]
        color = '#3498db' if polarity == '+' else '#e74c3c'
        dash = 'solid' if polarity == '+' else 'dash'
        
        # 화살표
        fig.add_annotation(
            x=x1, y=y1, ax=x0, ay=y0,
            xref='x', yref='y', axref='x', ayref='y',
            showarrow=True, arrowhead=2, arrowsize=1.5,
            arrowcolor=color, arrowwidth=2
        )
        
        # 극성 레이블
        mx, my = (x0 + x1) / 2, (y0 + y1) / 2
        fig.add_annotation(
            x=mx, y=my, text=f'({polarity})',
            showarrow=False, font=dict(size=12, color=color, family='Arial Black')
        )
    
    # 노드 그리기
    node_x = [pos[0] for pos in nodes.values()]
    node_y = [pos[1] for pos in nodes.values()]
    node_text = list(nodes.keys())
    
    fig.add_trace(go.Scatter(
        x=node_x, y=node_y, mode='markers+text',
        marker=dict(size=30, color='#ecf0f1', line=dict(width=2, color='#2c3e50')),
        text=node_text, textposition='top center',
        textfont=dict(size=11), showlegend=False
    ))
    
    # 루프 레이블
    if loop_labels:
        for lx, ly, lt in loop_labels:
            fig.add_annotation(
                x=lx, y=ly, text=f'<b>{lt}</b>',
                showarrow=False,
                font=dict(size=14, color='#8e44ad'),
                bgcolor='rgba(155, 89, 182, 0.1)',
                bordercolor='#8e44ad', borderwidth=1, borderpad=4
            )
    
    fig.update_layout(
        title=title, height=500,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
    )
    return fig

# SaaS 스타트업 CLD
nodes = {
    'Customers': (2, 4),
    'Word-of-Mouth': (4, 5),
    'New Customers': (4, 3),
    'Server Load': (0, 3),
    'Service Quality': (0, 1.5),
    'Satisfaction': (1, 0.5),
    'Churn': (3, 0.5),
    'Investment': (-1, 2),
    'Capacity': (-1, 3.5)
}

edges = [
    # R1: Growth Engine
    ('Customers', 'Word-of-Mouth', '+'),
    ('Word-of-Mouth', 'New Customers', '+'),
    ('New Customers', 'Customers', '+'),
    # B1: Growth Limit
    ('Customers', 'Server Load', '+'),
    ('Server Load', 'Service Quality', '-'),
    ('Service Quality', 'Satisfaction', '+'),
    ('Satisfaction', 'Churn', '-'),
    ('Churn', 'Customers', '-'),
    # B2: Investment Response
    ('Server Load', 'Investment', '+'),
    ('Investment', 'Capacity', '+'),
    ('Capacity', 'Server Load', '-'),
]

loop_labels = [
    (3.5, 4.2, 'R1: Growth Engine'),
    (1.5, 1.8, 'B1: Growth Limit'),
    (-0.8, 3, 'B2: Investment')
]

fig = draw_cld(nodes, edges, 'SaaS Startup Growth: Causal Loop Diagram (CLD)', loop_labels)
fig.show()

print("💡 CLD 해석:")
print("   R1(성장 엔진): 고객 → 구전 → 신규고객 → 고객 (양의 극성만 → 강화 루프)")
print("   B1(성장 한계): 고객 → 부하 → 품질↓ → 만족↓ → 이탈↑ → 고객↓ (음의 극성 3개 → 균형 루프)")
print("   B2(투자 대응): 부하 → 투자 → 용량↑ → 부하↓ (음의 극성 1개 → 균형 루프)")
print("\n⚠️ 핵심 질문: B2(투자)가 B1(성장 한계)을 충분히 상쇄할 수 있는가?")

💡 CLD 해석:
   R1(성장 엔진): 고객 → 구전 → 신규고객 → 고객 (양의 극성만 → 강화 루프)
   B1(성장 한계): 고객 → 부하 → 품질↓ → 만족↓ → 이탈↑ → 고객↓ (음의 극성 3개 → 균형 루프)
   B2(투자 대응): 부하 → 투자 → 용량↑ → 부하↓ (음의 극성 1개 → 균형 루프)

⚠️ 핵심 질문: B2(투자)가 B1(성장 한계)을 충분히 상쇄할 수 있는가?


### 📝 이론 과제 6-2 (15분)

#### 과제: CLD 작성 연습

**질문:**

1. **스톡과 플로우**를 다음 세 가지 상황에 적용하시오. 각각에서 스톡은 무엇이고, 유입 플로우와 유출 플로우는 무엇인가? (각 1-2문장)
   - (a) 대학의 학생 수
   - (b) 기업의 브랜드 신뢰도
   - (c) 도시의 대기 오염 수준

2. **루프 유형 판별 연습**: 다음 루프의 유형(R 또는 B)을 판별하고, 그 근거를 쓰시오.
   - (a) 광고 → (+) → 매출 → (+) → 이익 → (+) → 광고 예산 → (+) → 광고
   - (b) 가격 인상 → (+) → 수익 → (+) → 품질 투자 → (+) → 고객 만족 → (-) → 가격 불만 → (-) → 수요

3. **채찍효과(Bullwhip Effect)**를 검색하고, 이것이 왜 **지연**이 있는 **균형 루프**의 문제인지 설명하시오. (3-4문장)

**조사 키워드:**
- "stock and flow examples"
- "bullwhip effect supply chain"

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

### ✅ 과제 6-2 제출란

**학번:** ___________
**이름:** ___________

#### 1. 스톡과 플로우 식별

**(a) 대학의 학생 수**
- 스톡: 
- 유입 플로우: 
- 유출 플로우: 

**(b) 기업의 브랜드 신뢰도**
- 스톡: 
- 유입 플로우: 
- 유출 플로우: 

**(c) 도시의 대기 오염 수준**
- 스톡: 
- 유입 플로우: 
- 유출 플로우: 

#### 2. 루프 유형 판별

**(a):** (R 또는 B) — 근거:

**(b):** (R 또는 B) — 근거:

#### 3. 채찍효과와 지연

(여기에 작성)

**출처:** (URL)

---

## ☕ 휴식 (15분)

---

---

## Part 3. 레버리지 포인트와 의도치 않은 결과

---

### 6.3.3 레버리지 포인트 (Leverage Point)

**레버리지 포인트**: 작은 개입으로 시스템 전체에 큰 변화를 일으킬 수 있는 지점

Donella Meadows의 12가지 레버리지 포인트 (효과 낮은 순 → 높은 순):

| 순위 | 레버리지 포인트 | 예시 | 효과 |
|------|----------------|------|------|
| 12 | 상수, 파라미터 | 금리 0.25% 인상 | ⭐ |
| 11 | 버퍼 크기 | 안전재고 수준 | ⭐ |
| 10 | 물질 흐름 구조 | 도로 확장 | ⭐ |
| 9 | 지연 | 정보 시스템 개선 | ⭐⭐ |
| 8 | 균형 루프 강도 | 규제 강화 | ⭐⭐ |
| 7 | 강화 루프 방향 | 인센티브 재설계 | ⭐⭐⭐ |
| 6 | 정보 흐름 | 투명성 강화 | ⭐⭐⭐ |
| 5 | 시스템 규칙 | 법규 변경 | ⭐⭐⭐⭐ |
| 4 | 자기조직화 | 혁신 허용 | ⭐⭐⭐⭐ |
| 3 | 시스템 목표 | 성장 → 지속가능성 | ⭐⭐⭐⭐⭐ |
| 2 | 패러다임 | 경쟁 → 협력 | ⭐⭐⭐⭐⭐ |
| 1 | 패러다임 초월 | 시스템 사고 자체 | ⭐⭐⭐⭐⭐ |

**핵심 통찰:** 사람들은 보통 12~10번(파라미터, 버퍼, 물리적 구조)에서 개입한다. 가장 쉽지만 **가장 효과가 낮다!**

In [7]:
# 시각화: 레버리지 포인트 효과 비교 (SaaS 예시)
leverage_data = pd.DataFrame({
    'Level': [
        'Parameter\n(Server +10%)',
        'Delay Reduction\n(Info System)',
        'Rule Change\n(Auto-Scaling)',
        'Goal Change\n(Sustainable Growth)',
        'Paradigm\n(Right-Sizing)'
    ],
    'Difficulty': [1, 3, 5, 7, 9],
    'Impact': [1, 3, 5, 8, 10],
    'Meadows_Rank': [12, 9, 5, 3, 2]
})

fig = go.Figure()

colors = ['#95a5a6', '#f39c12', '#e67e22', '#e74c3c', '#8e44ad']

fig.add_trace(go.Scatter(
    x=leverage_data['Difficulty'],
    y=leverage_data['Impact'],
    mode='markers+text',
    marker=dict(size=leverage_data['Impact']*8, color=colors, opacity=0.8,
                line=dict(width=2, color='white')),
    text=leverage_data['Level'],
    textposition='top center',
    textfont=dict(size=10),
    hovertemplate='Difficulty: %{x}<br>Impact: %{y}<br>%{text}<extra></extra>'
))

# 대각선 참조선
fig.add_trace(go.Scatter(
    x=[0, 10], y=[0, 10],
    mode='lines', line=dict(color='lightgray', dash='dot'),
    showlegend=False
))

# 영역 주석
fig.add_annotation(x=2, y=8, text='High Leverage Zone<br>(Hard but Effective)',
                   showarrow=False, font=dict(size=11, color='green'),
                   bgcolor='rgba(46,204,113,0.1)')
fig.add_annotation(x=8, y=2, text='Low Leverage Zone<br>(Easy but Weak)',
                   showarrow=False, font=dict(size=11, color='gray'),
                   bgcolor='rgba(149,165,166,0.1)')

fig.update_layout(
    title='Leverage Points: Difficulty vs Impact (SaaS Example)',
    xaxis_title='Implementation Difficulty',
    yaxis_title='System Impact',
    height=500,
    xaxis=dict(range=[0, 11]),
    yaxis=dict(range=[0, 11])
)
fig.show()

print("💡 레버리지 포인트 핵심:")
print("   - 서버 용량 10% 증가 (파라미터): 쉽지만 효과 미미")
print("   - 자동 스케일링 정책 (규칙 변경): 중간 난이도, 상당한 효과")
print("   - '지속 가능한 성장' 목표 전환: 어렵지만 근본적 변화")
print("\n📌 Meadows: '사람들은 가장 쉬운 곳(파라미터)에서 개입하지만, 진정한 변화는 목표와 패러다임에서 온다'")

💡 레버리지 포인트 핵심:
   - 서버 용량 10% 증가 (파라미터): 쉽지만 효과 미미
   - 자동 스케일링 정책 (규칙 변경): 중간 난이도, 상당한 효과
   - '지속 가능한 성장' 목표 전환: 어렵지만 근본적 변화

📌 Meadows: '사람들은 가장 쉬운 곳(파라미터)에서 개입하지만, 진정한 변화는 목표와 패러다임에서 온다'


### 6.3.4 의도치 않은 결과 (Unintended Consequences)

CLD의 가장 큰 가치: 정책 개입의 **의도치 않은 결과**를 예측하는 것

#### 코브라 효과 (Cobra Effect)

영국 식민지 인도에서 코브라를 줄이기 위해 보상금 정책 시행:

```
의도한 루프:     보상금 → 코브라 사냥↑ → 야생 코브라↓  ✓
의도하지 않은 루프: 보상금 → 코브라 사육↑ → 총 코브라↑  ✗
보상금 폐지 후:   사육 코브라 방출 → 코브라 더 증가!    ✗✗
```

#### 정책 저항 (Policy Resistance)

시스템의 균형 루프가 정책을 상쇄하는 현상:
- 빈곤 감소 현금 지원 → 소득 증가 → **자격 상실** → 지원 중단 → 소득 원복
- 빈곤 감소 현금 지원 → 소득 증가 → **임대료 인상** → 실질 소득 변화 없음

### 6.5.4 5장 DAG vs 6장 CLD

| 구분 | DAG (5장) | CLD (6장) |
|------|----------|----------|
| 구조 | 비순환 (Acyclic) | **순환 허용** |
| 피드백 | 표현 불가 | **핵심 요소** |
| 시간 | 정적 | **동적** |
| 주요 용도 | 인과 효과 추정 | 시스템 동태 이해 |
| 적합한 상황 | 마케팅 캠페인 효과 | 장기 정책 부작용 |

In [8]:
# 시각화: 코브라 효과 — 의도한 루프 vs 의도하지 않은 루프
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Intended Effect', 'Unintended Effect (Cobra Effect)'),
    horizontal_spacing=0.15
)

# 의도한 효과: 보상금 → 사냥↑ → 코브라↓
intended_x = [0, 1, 2]
intended_y = [1, 1, 1]
intended_labels = ['Bounty<br>Policy', 'Cobra<br>Hunting ↑', 'Wild<br>Cobras ↓']
intended_colors = ['#3498db', '#2ecc71', '#2ecc71']

fig.add_trace(go.Scatter(
    x=intended_x, y=intended_y, mode='markers+text',
    marker=dict(size=45, color=intended_colors, opacity=0.8),
    text=intended_labels, textposition='bottom center',
    textfont=dict(size=10), showlegend=False
), row=1, col=1)

for i in range(2):
    fig.add_annotation(
        x=intended_x[i+1]-0.12, y=1, ax=intended_x[i]+0.12, ay=1,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowsize=1.5, arrowcolor='#2ecc71'
    )

# 의도하지 않은 효과
# 위쪽 경로: 보상금 → 사육 → 코브라 증가
unintended_nodes = {
    'Bounty': (0, 1.5),
    'Farming': (1, 2.5),
    'Release': (2, 2.5),
    'Hunting': (1, 0.5),
    'Total Cobras': (2.5, 1.5)
}

for name, (x, y) in unintended_nodes.items():
    color = '#e74c3c' if name in ['Farming', 'Release', 'Total Cobras'] else '#3498db'
    fig.add_trace(go.Scatter(
        x=[x], y=[y], mode='markers+text',
        marker=dict(size=40, color=color, opacity=0.8),
        text=[name.replace(' ', '<br>')], textposition='bottom center',
        textfont=dict(size=10), showlegend=False
    ), row=1, col=2)

# 화살표
arrow_pairs = [
    ('Bounty', 'Farming', '#e74c3c'),
    ('Farming', 'Release', '#e74c3c'),
    ('Release', 'Total Cobras', '#e74c3c'),
    ('Bounty', 'Hunting', '#2ecc71'),
    ('Hunting', 'Total Cobras', '#2ecc71')
]

for src, tgt, color in arrow_pairs:
    x0, y0 = unintended_nodes[src]
    x1, y1 = unintended_nodes[tgt]
    fig.add_annotation(
        x=x1, y=y1, ax=x0, ay=y0,
        xref='x2', yref='y2', axref='x2', ayref='y2',
        showarrow=True, arrowhead=2, arrowsize=1.5, arrowcolor=color, arrowwidth=2
    )

fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False)
fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False)
fig.update_layout(
    height=400,
    title_text='Cobra Effect: Intended vs Unintended Consequences'
)
fig.show()

print("💡 코브라 효과의 교훈:")
print("   - 정책 입안자는 의도한 루프(녹색)만 생각했다")
print("   - 인센티브에 대한 전략적 대응(빨간색) = 의도하지 않은 루프")
print("   - CLD를 미리 작성했다면 이 부작용을 예측할 수 있었다")
print("\n📌 시스템 사고의 핵심: '이 정책이 어떤 새로운 피드백 루프를 만들 것인가?'")

💡 코브라 효과의 교훈:
   - 정책 입안자는 의도한 루프(녹색)만 생각했다
   - 인센티브에 대한 전략적 대응(빨간색) = 의도하지 않은 루프
   - CLD를 미리 작성했다면 이 부작용을 예측할 수 있었다

📌 시스템 사고의 핵심: '이 정책이 어떤 새로운 피드백 루프를 만들 것인가?'


### 📝 이론 과제 6-3 (15분)

#### 과제: 레버리지 포인트와 의도치 않은 결과

**질문:**

1. Donella Meadows의 **레버리지 포인트 12단계** 중, "정보 흐름(6위)"과 "시스템 규칙(5위)"의 차이를 구체적 예시와 함께 설명하시오. (3-4문장)

2. **코브라 효과**와 유사한 정책 실패 사례를 하나 검색하시오. 어떤 "의도하지 않은 피드백 루프"가 정책을 무력화했는지 CLD 형태로 설명하시오. (4-5문장)

3. 5장의 **DAG**와 6장의 **CLD**를 각각 어떤 상황에서 사용해야 하는지, 구체적 비즈니스 사례와 함께 비교하시오. (3-4문장)

**조사 키워드:**
- "Donella Meadows leverage points"
- "cobra effect policy failure examples"
- "perverse incentive examples"

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

### ✅ 과제 6-3 제출란

**학번:** ___________
**이름:** ___________

#### 1. 정보 흐름 vs 시스템 규칙

(여기에 작성)

#### 2. 코브라 효과 유사 사례

(여기에 작성)

#### 3. DAG vs CLD 사용 상황 비교

(여기에 작성)

**출처:** (URL)

---

## Part 4. 실습: CLD 시각화와 루프 분석

---

### 실습 4-1: CLD 작성 함수와 루프 판별

인과 루프 다이어그램을 코드로 정의하고, 루프 유형을 자동으로 판별하는 함수를 작성합니다.

In [9]:
# CLD 분석 도구 함수

def identify_loop_type(polarities):
    """루프 유형 식별
    
    Args:
        polarities: 극성 리스트 ('+' 또는 '-')
    Returns:
        str: 'R (Reinforcing)' 또는 'B (Balancing)'
    """
    negative_count = polarities.count('-')
    if negative_count % 2 == 0:
        return 'R (Reinforcing)'
    else:
        return 'B (Balancing)'

def analyze_cld(edges, name="CLD"):
    """CLD 구조 분석: 루프 식별 및 레버리지 포인트 후보
    
    Args:
        edges: list of (source, target, polarity)
        name: CLD 이름
    """
    # 변수별 연결 수 계산 (중심성)
    node_connections = {}
    for src, tgt, pol in edges:
        node_connections[src] = node_connections.get(src, 0) + 1
        node_connections[tgt] = node_connections.get(tgt, 0) + 1
    
    # 극성 통계
    positive = sum(1 for _, _, p in edges if p == '+')
    negative = sum(1 for _, _, p in edges if p == '-')
    
    print(f"\n{'='*50}")
    print(f"CLD 분석: {name}")
    print(f"{'='*50}")
    print(f"변수(노드) 수: {len(node_connections)}")
    print(f"연결(엣지) 수: {len(edges)}")
    print(f"양의 극성(+): {positive}개")
    print(f"음의 극성(-): {negative}개")
    
    # 레버리지 포인트 후보 (연결이 많은 변수)
    sorted_nodes = sorted(node_connections.items(), key=lambda x: x[1], reverse=True)
    print(f"\n[레버리지 포인트 후보 (연결 수 기준)]")
    for node, count in sorted_nodes[:5]:
        print(f"  {node}: {count}개 연결")
    
    return node_connections

# 루프 판별 테스트
print("[루프 유형 판별 테스트]")
print(f"  (+)(+)(+) → {identify_loop_type(['+', '+', '+'])}")
print(f"  (+)(-)(+) → {identify_loop_type(['+', '-', '+'])}")
print(f"  (+)(-)(-)(+) → {identify_loop_type(['+', '-', '-', '+'])}")
print(f"  (+)(-)(-)(-) → {identify_loop_type(['+', '-', '-', '-'])}")

print("\n💡 규칙: 음의 극성(-)이 짝수개면 R(강화), 홀수개면 B(균형)")

[루프 유형 판별 테스트]
  (+)(+)(+) → R (Reinforcing)
  (+)(-)(+) → B (Balancing)
  (+)(-)(-)(+) → R (Reinforcing)
  (+)(-)(-)(-) → B (Balancing)

💡 규칙: 음의 극성(-)이 짝수개면 R(강화), 홀수개면 B(균형)


In [10]:
# 실습: 조직 변화 CLD 분석
org_change_edges = [
    # R: 변화 모멘텀
    ('Change Success', 'Trust in Change', '+'),
    ('Trust in Change', 'Participation', '+'),
    ('Participation', 'Change Success', '+'),
    
    # B1: 변화 저항
    ('Change Speed', 'Uncertainty', '+'),
    ('Uncertainty', 'Resistance', '+'),
    ('Resistance', 'Change Speed', '-'),
    
    # B2: 변화 피로
    ('Change Speed', 'Workload', '+'),
    ('Workload', 'Fatigue', '+'),
    ('Fatigue', 'Participation', '-'),
    
    # 리더십 개입
    ('Resistance', 'Leadership Action', '+'),
    ('Leadership Action', 'Communication', '+'),
    ('Communication', 'Uncertainty', '-'),
]

# CLD 분석
analyze_cld(org_change_edges, "Organizational Change Dynamics")

# 루프별 유형 판별
print("\n[루프별 유형 판별]")
loops = {
    'R1 (Success Momentum)': ['+', '+', '+'],
    'B1 (Resistance)': ['+', '+', '-'],
    'B2 (Fatigue)': ['+', '+', '-'],
    'Leadership Loop': ['+', '+', '-']  # Resistance→Leadership→Communication→Uncertainty(-)
}
for name, pols in loops.items():
    print(f"  {name}: {identify_loop_type(pols)} (극성: {pols})")


CLD 분석: Organizational Change Dynamics
변수(노드) 수: 10
연결(엣지) 수: 12
양의 극성(+): 9개
음의 극성(-): 3개

[레버리지 포인트 후보 (연결 수 기준)]
  Participation: 3개 연결
  Change Speed: 3개 연결
  Uncertainty: 3개 연결
  Resistance: 3개 연결
  Change Success: 2개 연결

[루프별 유형 판별]
  R1 (Success Momentum): R (Reinforcing) (극성: ['+', '+', '+'])
  B1 (Resistance): B (Balancing) (극성: ['+', '+', '-'])
  B2 (Fatigue): B (Balancing) (극성: ['+', '+', '-'])
  Leadership Loop: B (Balancing) (극성: ['+', '+', '-'])


In [11]:
# 조직 변화 CLD 시각화
org_nodes = {
    'Change Success': (2, 4),
    'Trust in Change': (4, 4),
    'Participation': (3, 2.5),
    'Change Speed': (0, 3),
    'Uncertainty': (0, 1.5),
    'Resistance': (1, 0.5),
    'Workload': (-1, 2),
    'Fatigue': (-1, 1),
    'Leadership Action': (2, 0.5),
    'Communication': (1, 1.5)
}

org_loop_labels = [
    (3, 3.5, 'R1: Success<br>Momentum'),
    (0.3, 2, 'B1: Resistance'),
    (-0.5, 1.8, 'B2: Fatigue'),
]

fig = draw_cld(org_nodes, org_change_edges,
               'Organizational Change: CLD', org_loop_labels)
fig.show()

print("💡 조직 변화 CLD 시사점:")
print("   - R1(성공 모멘텀): 작은 성공 → 신뢰 → 참여 → 더 큰 성공")
print("   - B1(저항): 변화 속도 ↑ → 불확실성 ↑ → 저항 ↑ → 속도 ↓")
print("   - B2(피로): 변화 속도 ↑ → 업무 부담 ↑ → 피로 → 참여 ↓")
print("\n📌 전략: Quick Win으로 R1 활성화 + 소통으로 B1 약화 + 적절한 속도로 B2 관리")

💡 조직 변화 CLD 시사점:
   - R1(성공 모멘텀): 작은 성공 → 신뢰 → 참여 → 더 큰 성공
   - B1(저항): 변화 속도 ↑ → 불확실성 ↑ → 저항 ↑ → 속도 ↓
   - B2(피로): 변화 속도 ↑ → 업무 부담 ↑ → 피로 → 참여 ↓

📌 전략: Quick Win으로 R1 활성화 + 소통으로 B1 약화 + 적절한 속도로 B2 관리


### 📝 실습 과제 6-4 (15분)

#### 과제: 나만의 CLD 작성

아래 코드의 TODO를 완성하여 **환경 규제의 역효과** CLD를 작성하시오.

**배경:** A 지역에서 공장 배출 규제를 강화하면, 생산 비용이 증가하여 B 지역으로 공장이 이전하고, B 지역 배출이 증가하여 전체 배출량은 변하지 않는다 (탄소 누출, Carbon Leakage).

In [12]:
# ========== 여기서부터 코드를 작성하세요 ==========

# TODO 1: 환경 규제 CLD의 엣지 정의
# 힌트: 다음 변수들을 사용하세요
#   'Regulation (Region A)', 'Production Cost (A)', 'Factory Relocation',
#   'Emission (Region A)', 'Emission (Region B)', 'Total Emission'
# 극성(+/-)을 잘 생각해보세요!

env_edges = [
    # TODO: ('변수1', '변수2', '극성') 형태로 작성
    # 규제 → 비용 → 이전 → A 배출 감소 / B 배출 증가
]

# TODO 2: 노드 위치 정의 (x, y 좌표)
env_nodes = {
    # TODO: 각 변수의 위치를 정의하세요
}

# TODO 3: CLD 분석 실행
# analyze_cld(env_edges, "Environmental Regulation Side Effects")

# TODO 4: CLD 시각화 (draw_cld 함수 활용)
# fig = draw_cld(env_nodes, env_edges, 'Environmental Regulation: Carbon Leakage CLD')
# fig.show()

# ========== 여기까지 ==========

---

## Part 5. 실습: SaaS 성장 시뮬레이션과 정책 비교

---

### 실습 5-1: SaaS 스타트업 성장 시뮬레이션

앞서 작성한 CLD를 정량적 **스톡-플로우 모델**로 전환하여 시뮬레이션합니다.

**모델 구조:**
- **스톡**: 고객 수, 서버 용량
- **플로우**: 신규 유입, 고객 이탈
- **보조 변수**: 서비스 품질, 구전 효과, 서버 부하

**핵심 수식:**

$$\text{New Customers} = \text{Base Acquisition} + \text{Customers} \times \text{WoM Rate} \times \text{Quality}$$

$$\text{Churned} = \text{Customers} \times (\text{Base Churn} + \text{Quality Sensitivity} \times (1 - \text{Quality}))$$

$$\text{Customers}(t) = \text{Customers}(t-1) + \text{New} - \text{Churned}$$

In [13]:
def simulate_saas_growth(months=36, strategy='none',
                          initial_customers=100, initial_capacity=200,
                          wom_rate=0.02, base_acquisition=5,
                          base_churn=0.03, quality_sensitivity=0.05,
                          capacity_threshold=0.8, quality_degradation=0.5,
                          capacity_per_invest=50, invest_delay=3):
    """SaaS 성장 시뮬레이션
    
    Args:
        strategy: 'none', 'reactive', 'proactive', 'aggressive'
    Returns:
        pd.DataFrame with simulation results
    """
    customers = initial_customers
    capacity = initial_capacity
    quality = 1.0
    pending_investments = []
    
    history = {'month': [], 'customers': [], 'capacity': [],
               'load_ratio': [], 'quality': [],
               'new_customers': [], 'churned': [], 'investment': []}
    
    for month in range(months):
        # 1. 지연된 투자 적용
        to_apply = [inv for inv in pending_investments if inv[0] <= month]
        for inv in to_apply:
            capacity += inv[1]
            pending_investments.remove(inv)
        
        # 2. 부하율
        load_ratio = customers / capacity if capacity > 0 else 1.0
        
        # 3. 품질 (부하 > 임계치면 저하)
        if load_ratio > capacity_threshold:
            loss = (load_ratio - capacity_threshold) * quality_degradation
            quality = max(0.1, quality - loss)
        else:
            quality = min(1.0, quality + 0.1)
        
        # 4. 신규 고객
        wom = customers * wom_rate * quality
        new_cust = base_acquisition + wom
        
        # 5. 이탈
        eff_churn = base_churn + quality_sensitivity * (1 - quality)
        churned = customers * eff_churn
        
        # 6. 업데이트
        customers = max(0, customers + new_cust - churned)
        
        # 7. 투자 결정
        investment = 0
        if strategy == 'reactive' and load_ratio > 0.7:
            investment = capacity_per_invest
            pending_investments.append((month + invest_delay, investment))
        elif strategy == 'proactive' and month > 0 and month % 6 == 0:
            investment = capacity_per_invest
            pending_investments.append((month + invest_delay, investment))
        elif strategy == 'aggressive' and month > 0 and month % 3 == 0:
            investment = capacity_per_invest
            pending_investments.append((month + invest_delay, investment))
        
        # 기록
        history['month'].append(month)
        history['customers'].append(customers)
        history['capacity'].append(capacity)
        history['load_ratio'].append(load_ratio)
        history['quality'].append(quality)
        history['new_customers'].append(new_cust)
        history['churned'].append(churned)
        history['investment'].append(investment)
    
    return pd.DataFrame(history)

print("✅ SaaS 성장 시뮬레이션 함수 정의 완료!")
print("\n파라미터 설명:")
print("  - wom_rate: 구전 효과율 (고객당 월 신규 유입)")
print("  - base_churn: 기본 월 이탈률")
print("  - quality_sensitivity: 품질 저하 시 추가 이탈률")
print("  - capacity_threshold: 부하 임계치 (초과 시 품질 저하)")
print("  - invest_delay: 투자 효과가 나타나기까지의 지연 (월)")

✅ SaaS 성장 시뮬레이션 함수 정의 완료!

파라미터 설명:
  - wom_rate: 구전 효과율 (고객당 월 신규 유입)
  - base_churn: 기본 월 이탈률
  - quality_sensitivity: 품질 저하 시 추가 이탈률
  - capacity_threshold: 부하 임계치 (초과 시 품질 저하)
  - invest_delay: 투자 효과가 나타나기까지의 지연 (월)


In [14]:
# 4가지 투자 전략 비교 시뮬레이션
strategies = ['none', 'reactive', 'proactive', 'aggressive']
labels = {'none': 'No Investment', 'reactive': 'Reactive',
          'proactive': 'Proactive (6-mo)', 'aggressive': 'Aggressive (3-mo)'}
colors = {'none': '#95a5a6', 'reactive': '#3498db',
          'proactive': '#2ecc71', 'aggressive': '#e74c3c'}

results = {s: simulate_saas_growth(strategy=s) for s in strategies}

# 4-패널 대시보드
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Customer Count', 'Service Quality',
                    'Server Load Ratio', 'Strategy Comparison')
)

for s in strategies:
    df = results[s]
    fig.add_trace(go.Scatter(
        x=df['month'], y=df['customers'], name=labels[s],
        line=dict(color=colors[s], width=2), legendgroup=s
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=df['month'], y=df['quality'], name=labels[s],
        line=dict(color=colors[s], width=2), legendgroup=s, showlegend=False
    ), row=1, col=2)
    fig.add_trace(go.Scatter(
        x=df['month'], y=df['load_ratio'], name=labels[s],
        line=dict(color=colors[s], width=2), legendgroup=s, showlegend=False
    ), row=2, col=1)

# 임계치 표시
fig.add_hline(y=0.8, line_dash='dash', line_color='orange',
              annotation_text='Threshold', row=2, col=1)

# 전략별 최종 성과 바 차트
final_data = []
for s in strategies:
    df = results[s]
    final_data.append({
        'Strategy': labels[s],
        'Final Customers': df['customers'].iloc[-1],
        'Avg Quality': df['quality'].mean(),
        'Total Investment': df['investment'].sum()
    })
final_df = pd.DataFrame(final_data)

fig.add_trace(go.Bar(
    x=final_df['Strategy'], y=final_df['Final Customers'],
    name='Final Customers', marker_color=[colors[s] for s in strategies],
    showlegend=False
), row=2, col=2)

fig.update_xaxes(title_text='Month', row=1, col=1)
fig.update_xaxes(title_text='Month', row=1, col=2)
fig.update_xaxes(title_text='Month', row=2, col=1)
fig.update_yaxes(title_text='Customers', row=1, col=1)
fig.update_yaxes(title_text='Quality (0-1)', row=1, col=2)
fig.update_yaxes(title_text='Load Ratio', row=2, col=1)
fig.update_yaxes(title_text='Final Customers', row=2, col=2)

fig.update_layout(height=600, title_text='SaaS Growth Simulation: 4 Investment Strategies (36 months)')
fig.show()

# 성과 요약 테이블
print("\n📊 투자 전략별 성과 비교 (36개월)")
print(f"{'전략':<20} {'최종고객':>10} {'평균품질':>10} {'누적투자':>10} {'투자효율':>10}")
print("-" * 65)
for s in strategies:
    df = results[s]
    final_cust = df['customers'].iloc[-1]
    avg_qual = df['quality'].mean()
    total_inv = df['investment'].sum()
    efficiency = final_cust / max(total_inv, 1)
    print(f"{labels[s]:<20} {final_cust:>10.0f} {avg_qual:>10.2f} {total_inv:>10.0f} {efficiency:>10.1f}")


📊 투자 전략별 성과 비교 (36개월)
전략                         최종고객       평균품질       누적투자       투자효율
-----------------------------------------------------------------
No Investment               157       0.84          0      157.1
Reactive                    221       1.00        150        1.5
Proactive (6-mo)            221       1.00        250        0.9
Aggressive (3-mo)           221       1.00        550        0.4


In [15]:
# 민감도 분석: 핵심 파라미터가 결과에 미치는 영향
sensitivity_results = []

# 1. 구전 효과율 변화
for wom in [0.01, 0.015, 0.02, 0.025, 0.03, 0.04]:
    df = simulate_saas_growth(strategy='proactive', wom_rate=wom)
    sensitivity_results.append({
        'Parameter': 'WoM Rate', 'Value': wom,
        'Final Customers': df['customers'].iloc[-1]
    })

# 2. 투자 지연 변화
for delay in [1, 2, 3, 4, 5, 7]:
    df = simulate_saas_growth(strategy='proactive', invest_delay=delay)
    sensitivity_results.append({
        'Parameter': 'Investment Delay', 'Value': delay,
        'Final Customers': df['customers'].iloc[-1]
    })

# 3. 품질 민감도 변화
for qs in [0.02, 0.03, 0.05, 0.07, 0.08, 0.10]:
    df = simulate_saas_growth(strategy='proactive', quality_sensitivity=qs)
    sensitivity_results.append({
        'Parameter': 'Quality Sensitivity', 'Value': qs,
        'Final Customers': df['customers'].iloc[-1]
    })

sens_df = pd.DataFrame(sensitivity_results)

# 시각화
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('WoM Rate', 'Investment Delay (months)', 'Quality Sensitivity')
)

params = ['WoM Rate', 'Investment Delay', 'Quality Sensitivity']
colors_s = ['#3498db', '#e74c3c', '#f39c12']

for i, param in enumerate(params):
    data = sens_df[sens_df['Parameter'] == param]
    fig.add_trace(go.Scatter(
        x=data['Value'], y=data['Final Customers'],
        mode='lines+markers', name=param,
        line=dict(color=colors_s[i], width=2),
        marker=dict(size=8)
    ), row=1, col=i+1)

fig.update_yaxes(title_text='Final Customers', col=1)
fig.update_layout(
    height=350,
    title_text='Sensitivity Analysis: Key Parameters Impact on Final Customers (Proactive Strategy)'
)
fig.show()

# 영향도 요약
print("\n📊 민감도 분석 결과 (선제적 투자 전략 기준)")
print("-" * 50)
for param in params:
    data = sens_df[sens_df['Parameter'] == param]
    min_val = data['Final Customers'].min()
    max_val = data['Final Customers'].max()
    impact = (max_val - min_val) / min_val * 100
    print(f"  {param}: 변동폭 ±{impact:.0f}% ({min_val:.0f} ~ {max_val:.0f}명)")

print("\n💡 시사점:")
print("   - 구전 효과율이 가장 큰 영향 → 고객 만족/추천 관리 핵심")
print("   - 투자 지연이 클수록 성과 급감 → 신속한 의사결정 필요")
print("   - 품질 민감도 높을수록 이탈 가속 → 품질 유지가 성장의 전제")


📊 민감도 분석 결과 (선제적 투자 전략 기준)
--------------------------------------------------
  WoM Rate: 변동폭 ±102% (178 ~ 358명)
  Investment Delay: 변동폭 ±0% (221 ~ 221명)
  Quality Sensitivity: 변동폭 ±0% (221 ~ 221명)

💡 시사점:
   - 구전 효과율이 가장 큰 영향 → 고객 만족/추천 관리 핵심
   - 투자 지연이 클수록 성과 급감 → 신속한 의사결정 필요
   - 품질 민감도 높을수록 이탈 가속 → 품질 유지가 성장의 전제


### 📝 실습 과제 6-5 (15분)

#### 과제: 시뮬레이션 파라미터 실험

아래 TODO를 완성하여 **다른 조건에서의 최적 전략**을 찾으시오.

In [16]:
# ========== 여기서부터 코드를 작성하세요 ==========

# 시나리오: 투자 지연이 6개월로 긴 경우 (기존 3개월 → 6개월)
# 질문: 투자 지연이 길어지면 어떤 전략이 가장 효과적인가?

# TODO 1: 4가지 전략 모두에 대해 invest_delay=6으로 시뮬레이션
long_delay_results = {}
for s in ['none', 'reactive', 'proactive', 'aggressive']:
    pass  # TODO: simulate_saas_growth(...) 호출

# TODO 2: 결과를 표로 출력 (최종 고객, 평균 품질, 누적 투자)

# TODO 3: 기본(지연 3개월) vs 긴 지연(6개월)에서 선제적 투자 전략의 성과 비교
# 힌트: results['proactive'] vs long_delay_results['proactive'] 비교

# TODO 4: 결과 해석 (print로 작성)
# - 지연이 길어지면 어떤 전략의 상대적 효과가 변하는가?
# - 비즈니스 함의는 무엇인가?

# ========== 여기까지 ==========

---

## 🎓 강의 마무리 및 핵심 요약

### 오늘 배운 내용

1. **선형적 사고의 한계**: "A→B"로 충분하지 않다. 피드백 루프, 지연, 비선형성이 만드는 동적 복잡성을 고려해야 한다

2. **두 가지 피드백 루프**: 강화 루프(R)는 변화를 증폭하고, 균형 루프(B)는 변화를 억제한다. 시스템의 동태는 이 둘의 상호작용으로 결정된다

3. **스톡과 플로우**: 스톡(축적)과 플로우(변화율)가 시스템의 골격이다. 스톡의 관성이 시스템의 "기억"을 만든다

4. **지연이 불안정을 만든다**: 지연이 길수록 오버슈팅과 진동이 발생한다. 선제적 대응이 반응적 대응보다 효과적인 이유

5. **레버리지 포인트는 파라미터가 아니다**: 예산/인력 조정은 쉽지만 효과가 낮다. 규칙, 목표, 패러다임을 바꾸는 것이 진정한 변화

### 핵심 메시지

> **기획자의 역할은 부분을 최적화하는 것이 아니라 전체 시스템을 이해하는 것이다.** 시스템 다이내믹스는 "왜 좋은 의도가 나쁜 결과를 만드는가"를 이해하게 해주며, 레버리지 포인트를 찾아 효과적으로 개입하게 해준다.

### 다음 장 예고

**제7장: 환경 분석** — 외부 환경의 동적 변화를 분석한다. 시스템 다이내믹스로 내부 구조를 이해했다면, 7장에서는 외부 환경(시장, 경쟁, 기술, 규제)의 변화를 체계적으로 분석하는 방법을 학습한다.

### 📚 추가 학습 자료

- Meadows, D.H. (2008). *Thinking in Systems: A Primer.* Chelsea Green Publishing.
- Senge, P.M. (1990). *The Fifth Discipline.* Doubleday.
- Sterman, J.D. (2000). *Business Dynamics.* McGraw-Hill.
- **Python 도구**: PySD (System Dynamics in Python), Vensim (전문 SD 소프트웨어)
- **온라인**: MIT System Dynamics Group (https://mitsloan.mit.edu/academic-departments/system-dynamics)

---

**제6장 끝**